# HumanEval dataset constructor

Builds an SFT-ready CSV from **humaneval_pipeline_output.csv** and the HumanEval dataset.

- **Input:** Upload `humaneval_pipeline_output.csv` in Colab, or set a local path.
- **Process:** Loads `openai/openai_humaneval`; uses extractors (no merges) to add `prompt` and reference solution. For **passed** rows the target is `generated_code`; for **hallucinated** rows the target is prefix (imports + function def) + normalized body from `canonical_solution` with proper 4-space indentation.
- **Output:** CSV with columns `dataset`, `task_id`, `prompt`, `canonical_solution`, `status`, `generated_code` for use with [SFT_LoRA_Adapters.ipynb](SFT_LoRA_Adapters.ipynb).

In [ ]:
import os
import sys
import pandas as pd

# Colab: upload CSV when prompted; local: use path below
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Upload humaneval_pipeline_output.csv")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0] if uploaded else ""
    OUT_DIR = "/content"
else:
    # Local: try pipeline folder, then cwd; output saved in cwd (run from FED/HumanEval to save there)
    cwd = os.getcwd()
    pipeline_folder = os.path.join(cwd, "Pipeline construction", "AST+DYNMAIC+LIB_API")
    if not os.path.isdir(pipeline_folder):
        pipeline_folder = os.path.join(cwd, "..", "Pipeline construction", "AST+DYNMAIC+LIB_API")
    CSV_PATH = os.path.join(pipeline_folder, "humaneval_pipeline_output.csv")
    if not os.path.isfile(CSV_PATH):
        CSV_PATH = os.path.join(cwd, "humaneval_pipeline_output.csv")
    OUT_DIR = cwd

OUT_CSV = os.path.join(OUT_DIR, "human_eval_sft_ready.csv")
if not CSV_PATH or not os.path.isfile(CSV_PATH):
    raise FileNotFoundError("Input CSV not found. In Colab, upload humaneval_pipeline_output.csv; locally set CSV_PATH.")
print("Input CSV:", CSV_PATH)
print("Output CSV:", OUT_CSV)

In [ ]:
# Load pipeline CSV (from file only; no merge)
df = pd.read_csv(CSV_PATH)
df["task_id"] = df["task_id"].astype(str)
print("Pipeline rows:", len(df))

# Load HumanEval and build lookup by task_id (no merge)
from datasets import load_dataset

ds_he = load_dataset("openai/openai_humaneval", split="test")
he_prompt_by_id = {row["task_id"]: row["prompt"] for row in ds_he}
he_canonical_by_id = {row["task_id"]: row["canonical_solution"] for row in ds_he}
print("HumanEval tasks in lookup:", len(he_prompt_by_id))

In [ ]:
def construct_prompt_humaneval(docstring_prompt):
    system_message = (
        "You are an expert Python developer. Your task is to complete the function "
        "provided by the user. Follow the docstring exactly. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Complete this Python function:\n{docstring_prompt}"}
    ]
    return messages

def messages_to_prompt_string(messages):
    parts = []
    for m in messages:
        role = m.get("role", "")
        content = m.get("content", "")
        parts.append(f"{role.capitalize()}: {content}")
    return "\n\n".join(parts)

def extract_prompt_for_task_id(task_id):
    return he_prompt_by_id.get(str(task_id), "")

def extract_canonical_for_task_id(task_id):
    return he_canonical_by_id.get(str(task_id), "")

def extract_imports_and_function_def(prompt_str):
    if not prompt_str or pd.isna(prompt_str):
        return ""
    s = str(prompt_str).strip()
    return s if not s.endswith("\n") else s.rstrip("\n") + "\n"

def normalize_body_indentation(body_str, indent_level=4):
    if not body_str or pd.isna(body_str):
        return ""
    lines = str(body_str).splitlines()
    if not lines:
        return ""
    non_empty = [ln for ln in lines if ln.strip()]
    if not non_empty:
        return ""
    min_indent = min(len(ln) - len(ln.lstrip()) for ln in non_empty)
    out = []
    for ln in lines:
        if not ln.strip():
            out.append("")
            continue
        current_indent = len(ln) - len(ln.lstrip())
        rel = max(0, current_indent - min_indent)
        content = (ln[min_indent:] if min_indent <= len(ln) else ln).lstrip()
        out.append(" " * (indent_level + rel) + content)
    return "\n".join(out)

print("Extractors and prompt builders defined.")

In [ ]:
# Add columns via extractors (no merge)
df["prompt_he"] = df["task_id"].apply(extract_prompt_for_task_id)
df["canonical_solution_he"] = df["task_id"].apply(extract_canonical_for_task_id)

# SFT prompt string (same format as prompt_extractor)
df["prompt"] = df["prompt_he"].apply(
    lambda p: messages_to_prompt_string(construct_prompt_humaneval(p)) if p else ""
)

# Build canonical_solution: passed -> generated_code; hallucinated -> prefix + normalized body
def build_canonical_solution(row):
    status = str(row.get("status", "")).strip().lower()
    if status == "passed":
        return str(row.get("generated_code", "") or "").strip()
    prefix = extract_imports_and_function_def(row.get("prompt_he", ""))
    body = str(row.get("canonical_solution_he", "") or "").strip()
    body = normalize_body_indentation(body, indent_level=4)
    if not prefix:
        return body
    return prefix.rstrip("\n") + "\n" + body

df["canonical_solution"] = df.apply(build_canonical_solution, axis=1)
print("Added prompt and canonical_solution.")

In [ ]:
# Output: dataset, task_id, prompt, canonical_solution, status, generated_code
df["dataset"] = "humaneval"
cols_out = ["dataset", "task_id", "prompt", "canonical_solution", "status", "generated_code"]
out_df = df[[c for c in cols_out if c in df.columns]].copy()

out_df.to_csv(OUT_CSV, index=False)
print(f"Saved {len(out_df)} rows to {OUT_CSV}")
print("Columns:", list(out_df.columns))
sample = out_df["canonical_solution"].iloc[0] if len(out_df) else ""; print("Sample canonical_solution (first 150 chars):", (str(sample)[:150] + "...") if sample else "(empty)")

if IN_COLAB:
    files.download(OUT_CSV)
    print("Download started: human_eval_sft_ready.csv")